Copied from https://www.kaggle.com/code/nihilisticneuralnet/44-50-got-lucky-but-not-for-long  
Reasoning with Multi Temperatures and min_p (4 groups) and strategy of score = Σ(entropy_scores) × vote_factor, which can likely achieve 10/10 in reference.csv (The hard problems are "86e8e5","dd7f5e").   
Current best scores of 41 in version 5, scores of 37 in version 3, scores of 36 in version4.
I found that my attempts (16 times) in pre versions (3, 4) were too time-limited (5 hours GPU times), although it could give a high possibility to get 10/10 in reference.csv.

In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [ ]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
import statistics
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
#config
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, methods, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show ALL intermediate results '
        'explicitly - every calculation, every step, every transformation.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n'
        '6. REFLECT: Review your entire solution. Question each step. Look for potential errors '
        'or oversights. If something seems wrong, restart with a different approach.\n\n'
        
        '# Critical Requirements:\n'
        '- WRITE DOWN every intermediate result explicitly. Do not skip steps.\n'
        '- SELF-CHECK after each major step. Ask yourself: "Is this correct? Does this make sense?"\n'
        '- REFLECT on your reasoning periodically. Look for logical gaps or computational errors.\n'
        '- If you find an error, acknowledge it and correct your approach.\n'
        '- Show your complete chain of reasoning - quality and thoroughness matter.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n'
        '- Double-check your final answer before submitting\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        '**You must put your final numerical answer inside \\boxed{}, e.g., \\boxed{42}**\n\n'
        
        'Think step-by-step and show your complete reasoning process. Write down ALL '
        'intermediate results. Quality and thoroughness '
        'of reasoning is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    min_completed_before_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    
    # sampling_groups = [
    #     # (1.1, 0.01),  
    #     (1.0, 0.02),  # 已知最佳
    #     # (0.9, 0.03),   
    #     # (0.8, 0.04),
    #     # (0.7, 0.05),
    #     # (0.6, 0.06),
    #     # (0.5, 0.07),
    #     # (0.4, 0.08),
    #     # (0.3, 0.09),
    # ]
    #use_topP
    sampling_groups = [
        (1.0, 0.95)
    ]

    use_topp = True
    
    # 答案评分参数（基于entropy）
    # score = Σ(exp(-entropy * scale)) × (1 + α × ln(1 + votes))
    score_entropy_scale = 2.0   # entropy缩放系数，越大对低entropy奖励越多
    score_vote_alpha = 0.15     # 投票调整系数，控制在15-20%差距内

In [ ]:
class ResultExtractor:
    '''提取模型输出中的结构化信息'''
    
    @staticmethod
    def _last_group(patterns, text: str, flags: int = 0) -> str | None:
        if isinstance(patterns, str):
            patterns = [patterns]
        for pattern in patterns:
            matches = re.findall(pattern, text, flags)
            if matches:
                last = matches[-1]
                if isinstance(last, tuple):
                    return last[-1]
                return last
        return None
    
    @staticmethod
    def extract_answer(text: str) -> int | None:
        '''提取\boxed{}中的答案'''
        patterns = [
            r'\\boxed\s*\{\s*([0-9,]+)\s*\}',
            r'final\s+answer\s+is\s*([0-9,]+)',
            r'answer\s*[:=]\s*([0-9,]+)'
        ]
        raw = ResultExtractor._last_group(patterns, text, flags=re.IGNORECASE)
        if raw:
            try:
                clean_value = raw.replace(',', '')
                value = int(clean_value)
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass
        return None

In [ ]:
class AnswerScorer:
    '''
    基于Entropy的答案评分系统
    
    设计原则：
    1. Entropy是核心：每次推理必定产生，低entropy=高确定性=更可能正确
    2. Vote是温和调整：用对数压缩，防止投票数主导结果
    '''
    
    def __init__(self, cfg):
        self.cfg = cfg
    
    def compute_entropy_score(self, entropy: float | None) -> float:
        '''
        基于entropy计算单个结果的得分
        
        entropy越低，得分越高：score = exp(-entropy * scale)
        - entropy=0.5, scale=2.0 → score≈0.37
        - entropy=1.0, scale=2.0 → score≈0.14
        - entropy=2.0, scale=2.0 → score≈0.02
        '''
        if entropy is None or not math.isfinite(entropy):
            return 0.01  # 无entropy信息时给极低分
        
        # 限制entropy范围防止数值问题
        entropy = max(0.0, min(entropy, 10.0))
        return math.exp(-entropy * self.cfg.score_entropy_scale)
    
    def compute_vote_factor(self, votes: int) -> float:
        '''
        计算投票调整因子（温和）
        
        使用对数压缩：factor = 1 + α × ln(1 + votes)
        - 1 vote: 1 + 0.15×ln(2) ≈ 1.10
        - 3 votes: 1 + 0.15×ln(4) ≈ 1.21
        - 6 votes: 1 + 0.15×ln(7) ≈ 1.29
        最大差距约17%，投票数不会主导结果
        '''
        return 1.0 + self.cfg.score_vote_alpha * math.log(1 + votes)
    
    def compute_answer_score(self, results_for_answer: list) -> float:
        '''
        计算某个答案的综合得分
        
        公式: score = Σ(entropy_scores) × vote_factor
        '''
        if not results_for_answer:
            return 0.0
        
        # 基础分：所有结果的entropy得分之和
        base_score = sum(
            self.compute_entropy_score(r.get('Entropy')) 
            for r in results_for_answer
        )
        
        # 温和的投票调整
        votes = len(results_for_answer)
        vote_factor = self.compute_vote_factor(votes)
        
        return base_score * vote_factor
    
    def select_best_answer(self, detailed_results: list) -> tuple[int, pd.DataFrame]:
        '''
        从所有结果中选出最佳答案
        
        评分完全基于entropy和vote
        '''
        # # 收集所有有答案的结果
        # valid_results = [r for r in detailed_results if r.get('Answer') is not None]
        
        # if not valid_results:
        #     return 0, pd.DataFrame()
        
        # # 按答案分组
        # answer_groups = defaultdict(list)
        # for r in valid_results:
        #     answer_groups[r['Answer']].append(r)
        
        # # 计算每个答案的得分
        # scored_answers = []
        # for answer, results in answer_groups.items():
        #     score = self.compute_answer_score(results)
        #     votes = len(results)
            
        #     # 计算统计信息
        #     entropies = [r.get('Entropy') for r in results if r.get('Entropy') is not None and math.isfinite(r.get('Entropy'))]
        #     avg_entropy = sum(entropies) / len(entropies) if entropies else float('nan')
            
        #     scored_answers.append({
        #         'Answer': answer,
        #         'Votes': votes,
        #         'Score': score,
        #         'AvgEntropy': avg_entropy
        #     })
        
        # # 按得分排序
        # scored_answers.sort(key=lambda x: x['Score'], reverse=True)
        
        # # 创建DataFrame
        # df = pd.DataFrame(scored_answers)
        # df = df.round({'Score': 4, 'AvgEntropy': 3})
        
        # best_answer = scored_answers[0]['Answer'] if scored_answers else 0
        
        # return best_answer, df
        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer, vote_dataframe

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:#用于输入模板搭建

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt) #system prompt
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH) #GPT-OSS low, medium, high
            .with_tools(tool_config) #有哪些工具、工具命名空间等
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [ ]:
'''
可控、可复用、带超时保护的 Python 执行沙箱，本质上是用 Jupyter Kernel 在后台跑代码，并把 stdout / stderr / 执行结果完整捕获返回给上层（例如 LLM）。

LLM → (生成 Python 代码)
      ↓
AIMO3Sandbox.execute()
      ↓
Jupyter Kernel (独立进程)
      ↓
stdout / stderr / result
      ↓
字符串形式返回给 LLM

'''
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 60000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        #连接端口
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False #防止多次申请
        self._client = None #Jupyter client（通信）
        self._km = None #KernelManager（生命周期）
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore' #防止 warnings 污染 stdout（对 LLM 极重要）
        env['MPLBACKEND'] = 'Agg' #禁止 GUI 后端, 防止 matplotlib 卡死

        self._km = KernelManager()#juypter需要五个端口，需要提前固定端口
        self._km.shell_port = ports[0] #执行命令
        self._km.iopub_port = ports[1] #stdout / stderr / result
        self._km.stdin_port = ports[2] #输入
        self._km.hb_port = ports[3] #心跳
        self._km.control_port = ports[4] #控制

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL']) #禁用日志输出

        self._client = self._km.blocking_client() #建立客户端通信
        self._client.start_channels() #等待 kernel 完全 ready
        self._client.wait_for_ready(timeout=self._default_timeout) #防止 race condition
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )#初始化内置数学环境

    def _format_error(self, traceback: list[str]) -> str:
        '''
        清洗 Jupyter traceback

        去掉 ANSI 颜色码

        去掉系统无关文件路径
        '''
        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        ) #执行代码，获取本次消息的确认id

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:#超时中断
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0) #IOPub 消息循环（Jupyter 协议）

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:#只处理属于本次执行的消息
                continue 

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream': #stdout / stderr
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error': #traceback
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}: #捕获代码执行结果
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status': 
                if content.get('execution_state') == 'idle': #执行完成
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None #是否有沙盒
        
        self._execution_lock = threading.Lock() #防止多段代码同时跑在同一个 kernel
        self._init_lock = threading.Lock() #防止并发创建多个 kernel

    def _ensure_session(self):

        if self._jupyter_session is None:#没有传入sandbox，创建
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:
        #确保LLM给的最后结果是print(2+2)而不是2+2
        lines = code.strip().split('\n')

        if not lines: #如果 code 原本就是空字符串或全是空白字符，strip() 后会变成空字符串
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line: ##如果最后一行是空行（去除空白后为空），返回原代码
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        #向 LLM 注册一个名为 python 的 tool
        #描述就是 tool_prompt
        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction,  #传入tool_prompt，告诉模型要用工具怎么做
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant') #recipient接受者

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session() #确保 sandbox 存在
        raw_script = message.content[0].text #取出 LLM 生成的代码
        final_script = self._ensure_last_print(raw_script) #自动补 print

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)#执行代码得到返回消息

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 50000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS) 
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions() 
        self.extractor = ResultExtractor()
        self.scorer = AnswerScorer(cfg)
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
        cmd = [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server', 
            '--seed', str(self.cfg.seed), 
            '--model', self.cfg.model_path, 
            '--served-model-name', self.cfg.served_model_name, 
            '--tensor-parallel-size', '1', 
            '--max-num-seqs', str(self.cfg.batch_size), 
            '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization), 
            '--host', '0.0.0.0', 
            '--port', str(self.port), 
            '--dtype', self.cfg.dtype, 
            '--kv-cache-dtype', self.cfg.kv_cache_dtype, 
            '--max-model-len', str(self.cfg.context_tokens), 
            '--stream-interval', str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)
    
    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
            if return_code is not None:
                self.log_file.flush()
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            if not isinstance(top_logprobs_dict, dict) or not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        return total_entropy / token_count if token_count > 0 else float('inf')
    
    @staticmethod
    def _is_valid_answer(answer) -> bool:
        if answer is None:
            return False
        if isinstance(answer, float) and math.isnan(answer):
            return False
        return True
    
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float,
        temperature: float,
        # min_p: float
        top_p: float
    ) -> dict:
        '''执行单次推理尝试，提取答案'''
        
        if stop_event.is_set() or time.time() > deadline:
            return self._empty_result(attempt_index)
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        full_response = []
        logprobs_buffer = []
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, problem, local_tool.tool_config
            )
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=temperature, 
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={
                        # 'min_p': min_p, 
                        'top_p': top_p,
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            full_response.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            if chunk_logprobs is not None and chunk_logprobs.top_logprobs:
                                logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        # 实时扫描答案
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self.extractor.extract_answer(search_text)
                            if answer is not None:
                                final_answer = answer
                                break
    
                finally:
                    stream.close()
    
                if final_answer is not None:
                    break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self.extractor.extract_answer(answer_text)
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)
                    response_text = tool_responses[0].content[0].text
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text:
                        python_errors += 1
                    conversation.messages.extend(tool_responses)
    
        except Exception:
            python_errors += 1
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
        
        # 如果没有在流中找到，最后再扫描一次完整响应
        if full_response and final_answer is None:
            full_text = ''.join(full_response)
            final_answer = self.extractor.extract_answer(full_text)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
    
        return {
            'Attempt': attempt_index + 1, 
            'Answer': final_answer,
            'Entropy': mean_entropy, 
            'Tokens': total_tokens, 
            'PyCalls': python_calls, 
            'PyErrors': python_errors
        }
    
    def _empty_result(self, attempt_index: int) -> dict:
        return {
            'Attempt': attempt_index + 1, 
            'Answer': None, 
            'Entropy': float('inf'),
            'Tokens': 0, 
            'PyCalls': 0, 
            'PyErrors': 0
        }
    
    def solve_problem(self, problem: str) -> int:
    
        print(f'\nProblem: {problem}\n')
        
        user_input = f'{problem} {self.cfg.preference_prompt}'
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
    
        detailed_results = []
        valid_answers = []
        completed_attempts = 0
        
        stop_event = threading.Event()
        
        sampling_groups = self.cfg.sampling_groups
        if not sampling_groups:
            sampling_groups = [(1.0, 0.95)]
        
        batch_tasks = []
        for attempt_index in range(self.cfg.attempts):
            # temp, min_p = sampling_groups[attempt_index % len(sampling_groups)]
            # batch_tasks.append((self.cfg.system_prompt, attempt_index, temp, min_p))
            temp, top_p = sampling_groups[attempt_index % len(sampling_groups)]
            batch_tasks.append((self.cfg.system_prompt, attempt_index, temp, top_p))
        
        print(f'--- Parallel Attempts: {self.cfg.attempts} | Groups: {len(sampling_groups)} ---')
        for i, (t, mp) in enumerate(sampling_groups, start=1):
            # print(f'Group {i}: T={t}, min_p={mp}')
            print(f'Group {i}: T={t}, top_p={mp}')
        
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [
                executor.submit(
                    self._process_attempt, user_input, sp, idx, stop_event, deadline, t, mp
                ) for sp, idx, t, mp in batch_tasks
            ]
            
            for future in as_completed(futures):
                try:
                    result = future.result()
                    
                    if self._is_valid_answer(result.get('Answer')):
                        valid_answers.append(result['Answer'])
                        completed_attempts += 1
                        detailed_results.append(result)
                    
                    # 早停检查（必须严格大于 early_stop）
                    vote_counter = Counter(valid_answers)
                    max_votes = max(vote_counter.values(), default=0)
                    
                    if (max_votes > self.cfg.early_stop and 
                        completed_attempts >= self.cfg.min_completed_before_stop):
                        stop_event.set()
                        break
                
                except Exception as exc:
                    print(f'Future failed: {exc}')
        
        self.problems_remaining = max(0, self.problems_remaining - 1)
    
        # 显示结果
        if detailed_results:
            df = pd.DataFrame(detailed_results)
            df['Entropy'] = df['Entropy'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df)
    
        if not valid_answers:
            print('\nNo valid answers found. Returning 0.\n')
            return 0
    
        # 使用评分系统选择答案
        final_answer, score_df = self.scorer.select_best_answer(detailed_results)
        
        if not score_df.empty:
            print('\nAnswer Scores:')
            display(score_df)
        
        print(f'\n✅ Final Answer: {final_answer}\n')
        return final_answer
    
    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
        if hasattr(self, 'log_file'):
            self.log_file.close()
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
                except Exception:
                    pass

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions, ground_truth
    id_value = id_.item(0)
    question_text = question.item(0)

    # if id_value not in ["86e8e5"]:#save的话要注释掉这里, "86e8e5","dd7f5e"
    #     return pl.DataFrame({'id': id_value, 'answer': ground_truth[id_value]})
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)

    # Store prediction
    predictions[id_value] = final_answer
    # Check accuracy if ground truth available
    total_count += 1
    if id_value in ground_truth:
        gt = ground_truth[id_value]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
# Load reference data and keep ground truth for accuracy calculation
df = pd.read_csv(
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
)

# Store ground truth answers for accuracy calculation (only in local mode)
ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# Create input file without answers
# df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

In [ ]:
# for i in range(10):#统计十次的预测准确性
#     print(f"#batch {i}#")
# inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

# if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
#     inference_server.serve()
    
# else:
#     inference_server.run_local_gateway(
#         # ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
#         ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv',)
#     )
#     if ground_truth and total_count > 0:
#         print("\n" + "=" * 50)
#         print("📊 FINAL ACCURACY SUMMARY")
#         print("=" * 50)
#         print(f"Correct: {correct_count}/{total_count}")
#         print(f"Accuracy: {100*correct_count/total_count:.1f}%")
#         print("=" * 50)
        
#         # Show details
#         print("\nDetails:")
#         for qid, pred in predictions.items():
#             if qid in ground_truth:
#                 gt = ground_truth[qid]
#                 status = "✅" if pred == gt else "❌"
#                 print(f"  {qid}: pred={pred}, gt={gt} {status}")

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )